In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from newspaper import Article
import spacy

print("All imports working ✅")

All imports working ✅


In [6]:
import requests
import re
from urllib.parse import urljoin

# Load NLP model
nlp = spacy.load("en_core_web_sm")

# -------- INPUT --------
seed_urls = [
    # Direct articles
    "https://www.boomlive.in/fact-check/viral-video-union-finance-minister-nirmala-sitharaman-investment-platform-claim-fact-check-30639",
    "https://digiteye.in/does-this-viral-clip-show-rajdeep-sardesai-and-fm-nirmala-sitharaman-on-guaranteed-returns-on-investments-fact-check/",
    "https://www.indiatoday.in/fact-check/story/this-mukesh-ambani-video-is-a-deepfake-it-sells-a-scam-2433006-2023-09-08",
    "https://www.indiatoday.in/fact-check/story/fact-check-another-day-another-mukesh-ambani-deepfake-selling-a-get-rich-quick-scheme-2514012-2024-03-12",
    "https://www.lighthousejournalism.com/viral/fact-check-fake-videos-claim-sudha-and-narayan-murthy-endorse-investment-scheme-3899/",
    "https://newsmeter.in/fact-check/fact-check-ajit-pawar-left-crores-in-will-earned-via-ai-investment-platform-no-facebook-ad-is-fake-763566",

    
    # Listing pages (bulk)
    "https://www.newschecker.in/posts/page/1/investment",
    "https://factly.in/?s=investment"
]

# -------- GET ARTICLE LINKS FROM LISTING --------
def get_article_links(page_url):
    try:
        res = requests.get(page_url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        links = []
        for a in soup.find_all("a", href=True):
            href = a["href"]

            if any(x in href for x in ["fact-check", "investment", "viral"]):
                if href.startswith("http"):
                    links.append(href)
                else:
                    links.append(urljoin(page_url, href))

        return list(set(links))
    except:
        return []

# -------- SCRAPER --------
def scrape_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()

        if len(article.text) < 200:
            raise Exception("Too short")

        return article.title, article.text
    except:
        try:
            res = requests.get(url, timeout=10)
            soup = BeautifulSoup(res.text, "html.parser")

            title = soup.find("h1")
            title = title.get_text().strip() if title else ""

            paragraphs = soup.find_all("p")
            text = " ".join([p.get_text() for p in paragraphs])

            return title, text
        except:
            return "", ""

# -------- NER: INSTITUTION --------
def extract_institution(text):
    doc = nlp(text)

    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG"]:
            return ent.text

    return "Unknown"

# -------- SCAM VECTOR --------
def detect_scam_vector(text):
    text = text.lower()

    if "youtube" in text:
        return "YouTube"
    elif "deepfake" in text or "ai video" in text:
        return "Deepfake Video"
    elif "video" in text:
        return "Video"
    elif "tweet" in text or "twitter" in text:
        return "Social Media"
    elif "facebook" in text:
        return "Social Media"
    elif "article" in text:
        return "News Article"
    else:
        return "Other"

# -------- MISINFO TYPE --------
def detect_misinfo_type(text):
    text = text.lower()

    if "loan" in text:
        return "Loan Fraud"
    elif "investment" in text or "profit" in text:
        return "Investment Scam"
    elif "crypto" in text:
        return "Crypto Scam"
    elif "resign" in text or "minister" in text:
        return "Political Misinformation"
    else:
        return "Fake News"

# -------- CORE CLAIM --------
def extract_claim(text):
    sentences = re.split(r'[.!?]', text)

    for s in sentences[:15]:
        s_clean = s.strip().lower()

        if any(word in s_clean for word in ["claim", "viral", "circulating", "post", "video"]):
            return "A false claim that " + s.strip()

    return "A false claim related to financial misinformation."

# -------- ORIGINAL POST --------
def extract_tweet(text):
    match = re.search(r"(#\w+.*)", text)
    return match.group(1) if match else text[:200]

# -------- IMAGE TEXT --------
def extract_image_text(text):
    return text[:400]

# -------- IMAGE URL --------
def extract_image_url(url):
    try:
        res = requests.get(url, timeout=10)
        soup = BeautifulSoup(res.text, "html.parser")

        img = soup.find("img")
        return img["src"] if img else ""
    except:
        return ""

# -------- MAIN PIPELINE --------
all_urls = []
failed_urls = []

for seed in seed_urls:
    seed = seed.strip()
    
    if "page" in seed or "?s=" in seed:
        print(f"🔎 Extracting links from: {seed}")
        links = get_article_links(seed)
        
        if links:
            all_urls.extend(links[:7]) # Limit to first 7 links per listing page
        else:
            print(f"⚠️ No links found in: {seed}")
    else:
        all_urls.append(seed)

# Remove duplicates safely
all_urls = list(dict.fromkeys(all_urls))

for url in all_urls:
    print(f"⚡ Processing: {url}")

    title, text = scrape_article(url)

    if not text or len(text) < 100:
        print(f"❌ Skipped (no content): {url}")
        failed_urls.append(url)
        continue

    row = {
        "Institution": extract_institution(text),
        "Scam_Vector": detect_scam_vector(text),
        "Misinfo_Type": detect_misinfo_type(text),
        "Core_Claim": extract_claim(text),
        "Original_Post": extract_tweet(text),
        "Image_Text": extract_image_text(text),
        "Image_URL": extract_image_url(url)
    }

    rows.append(row)

# -------- SAVE --------
df = pd.DataFrame(rows)
df.to_csv("financial_misinfo_dataset.csv", index=False)

print("\n✅ DONE — Dataset saved as financial_misinfo_dataset.csv")
print("\n✅ SUCCESSFUL:", len(rows))
print("❌ FAILED:", len(failed_urls))

if failed_urls:
    print("\nFailed URLs:")
    for u in failed_urls:
        print(u)

🔎 Extracting links from: https://www.newschecker.in/posts/page/1/investment
🔎 Extracting links from: https://factly.in/?s=investment
⚡ Processing: https://www.boomlive.in/fact-check/viral-video-union-finance-minister-nirmala-sitharaman-investment-platform-claim-fact-check-30639
⚡ Processing: https://digiteye.in/does-this-viral-clip-show-rajdeep-sardesai-and-fm-nirmala-sitharaman-on-guaranteed-returns-on-investments-fact-check/
⚡ Processing: https://www.indiatoday.in/fact-check/story/this-mukesh-ambani-video-is-a-deepfake-it-sells-a-scam-2433006-2023-09-08
⚡ Processing: https://www.indiatoday.in/fact-check/story/fact-check-another-day-another-mukesh-ambani-deepfake-selling-a-get-rich-quick-scheme-2514012-2024-03-12
⚡ Processing: https://www.lighthousejournalism.com/viral/fact-check-fake-videos-claim-sudha-and-narayan-murthy-endorse-investment-scheme-3899/
⚡ Processing: https://newsmeter.in/fact-check/fact-check-ajit-pawar-left-crores-in-will-earned-via-ai-investment-platform-no-facebook